# PINN — Boussinesq Dispersive Shallow Water System
## Benchmark Cases from Adytia et al. (2019), *Computational Geosciences*

**Equations (weakly nonlinear VBM, eqs. 13–15):**

$$\partial_t \eta = -\partial_x(hu) - \partial_x(\beta \cdot \partial_x \Psi) \tag{13}$$
$$\partial_t u = -g\partial_x \eta - u\partial_x u - R_B \tag{14}$$
$$-\partial_x(\alpha \partial_x \Psi) + \gamma \Psi = \partial_x(\beta u) \tag{15}$$

with $h = d(x) + \eta$ and $\alpha,\beta,\gamma$ the vertical moments of the VBM profile,
evaluated at the local depth (§4).

**Available benchmark cases:**

| Key | Case | Reference | Verification |
|-----|------|-----------|--------------|
| `carrier_greenspan` | Regular wave run-up on a plane beach | Carrier & Greenspan (1958) | **Exact nonlinear solution** → RMSE/corr |
| `solitary_nonbreaking` | Solitary run-up H/d=0.0185 | Synolakis (1987) | Snapshots — needs digitised lab data |
| `solitary_breaking` | Solitary run-up H/d=0.3 | Synolakis (1987) | Snapshots — **no breaker model implemented** |
| `beji_battjes` | Harmonic wave over bar | Beji & Battjes (1993) | Gauge signals — **no sponge layer yet** |
| `flat_cosine` | Cosine wave flat bottom | — | Sanity check |

**Carrier–Greenspan is the only case with a true reference solution.** It is solved exactly
via the hodograph transformation (§5b), used to drive the offshore boundary and to set the
initial condition, and then compared against the PINN in §17. The other cases run, but their
"validation" is currently visual only.

> **Scope note.** CG solves the *non-dispersive* nonlinear shallow water equations; the VBM is
> dispersive. At this configuration $kd \approx 0.14$ offshore, so the two differ by
> $O((kd)^2/3) \approx 0.7\%$ — that is the noise floor of the comparison, not machine precision.

## 0. Install Dependencies

In [ ]:
# Uncomment if needed
# !pip install tensorflow>=2.12.0 numpy>=1.23.0 matplotlib>=3.6.0 scipy>=1.10.0

## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time
import os

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. ★ Select Benchmark Case Here ★

In [ ]:
# ============================================================
#  CHANGE THIS LINE TO SWITCH CASE
# ============================================================
ACTIVE_CASE = "carrier_greenspan"
# Options:
#   "carrier_greenspan"     — analytic solution available (start here)
#   "solitary_nonbreaking"  — Synolakis H/d = 0.0185
#   "solitary_breaking"     — Synolakis H/d = 0.3
#   "beji_battjes"          — harmonic wave over bar
#   "flat_cosine"           — generic sanity check
print(f"Active case: {ACTIVE_CASE}")

## 3. Case Configurations

In [ ]:
CASES = {

    # ------------------------------------------------------------------
    # CASE 1: Carrier & Greenspan (1958) — §4.2
    # Regular harmonic wave run-up on a PLANE sloping beach.
    #
    # Geometry note: the domain now starts exactly at the slope toe, so the
    # bed is a uniform slope everywhere.  The CG exact solution assumes a
    # plane beach — with the old flat shelf from x=-50 to x=-5.5 in front of
    # it, no closed-form reference exists for this geometry.
    #
    # Amplitude note: CG is single-valued (non-breaking) only while
    # A·κ̂³ ≤ 2, equivalently R ≤ g·s²/ω².  Here that caps the offshore
    # amplitude at ≈0.0119 m; the old 0.03 m is ~2.6× past breaking.
    # §5b asserts this, so a bad value fails loudly instead of silently
    # producing a multivalued "solution".
    # ------------------------------------------------------------------
    "carrier_greenspan": {
        "label"       : "Carrier & Greenspan (1958) — Regular Wave Run-up",
        "ref"         : "Carrier, G. & Greenspan, H. (1958). J. Fluid Mech. 4(1), 97–109",
        "g"           : 9.81,
        "d0"          : 0.5,       # depth at the offshore boundary x_min
        "kappa1_d"    : 1.52,      # optimal κ₁·d₀ (α, β, γ derived — see §4)
        "R_B"         : 0.0,       # breaking term (no breaker model implemented)
        "x_min"       : -5.5,      # = slope_start: plane beach, no shelf
        "x_max"       : 10.0,      # still-water shoreline is at x = 7.0
        "t_min"       : 0.0,
        "t_max"       : 40.0,      # 4 wave periods
        "ic_type"     : "carrier_greenspan",
        "wave_amp"    : 0.008,     # η amplitude at x_min  (must be ≤ ~0.0119)
        "wave_period" : 10.0,
        "bottom_type" : "sloping",
        "slope"       : 1.0 / 25.0,
        "slope_start" : -5.5,
        "bc_type"     : "cg_influx",
        "N_F"         : 10_000,
        "N_IC"        : 500,
        "N_BC"        : 400,
        "layers"      : [2, 64, 64, 64, 64, 3],
        "activation"  : "tanh",
        "lr_adam"     : 1e-3,
        "epochs_adam" : 20_000,
        "epochs_lbfgs": 3_000,
        "w_pde"       : 1.0,
        "w_ell"       : 1.0,
        "w_ic"        : 10.0,
        "w_bc"        : 10.0,
        "snap_times"  : [32.5, 35.0, 37.5, 40.0],   # one period of the run-up cycle
        "plot_freq"   : 1_000,
    },

    # ------------------------------------------------------------------
    # CASE 2: Solitary wave run-up — non-breaking (Synolakis 1987) — §4.1
    # H/d = 0.0185, d0 = 1.0 m
    # ------------------------------------------------------------------
    "solitary_nonbreaking": {
        "label"       : "Synolakis (1987) — Solitary Wave Run-up (Non-breaking)",
        "ref"         : "Synolakis, C.E. (1987). J. Fluid Mech. 185, 523–545",
        "g"           : 9.81,
        "d0"          : 1.0,
        "kappa1_d"    : 1.52,
        "R_B"         : 0.0,
        "x_min"       : -100.0,
        "x_max"       : 10.0,
        "t_min"       : 0.0,
        "t_max"       : 35.0,
        "ic_type"     : "solitary",
        "wave_height" : 0.0185,
        "wave_center" : -40.0,
        "bottom_type" : "sloping",
        "slope"       : 1.0 / 19.85,
        "slope_start" : -19.85,
        "bc_type"     : "wall_right",
        "N_F"         : 12_000,
        "N_IC"        : 600,
        "N_BC"        : 300,
        "layers"      : [2, 64, 64, 64, 64, 3],
        "activation"  : "tanh",
        "lr_adam"     : 1e-3,
        "epochs_adam" : 8_000,
        "epochs_lbfgs": 3_000,
        "w_pde"       : 1.0,
        "w_ell"       : 1.0,
        "w_ic"        : 10.0,
        "w_bc"        : 5.0,
        "snap_times"  : [12.83, 16.03, 19.24, 22.45],
        "plot_freq"   : 500,
    },

    # ------------------------------------------------------------------
    # CASE 3: Solitary wave run-up — breaking (Synolakis 1987) — §4.1
    # H/d = 0.3, d0 = 1.0 m
    # NOTE: R_B is still 0 — this runs WITHOUT a breaker model.
    # ------------------------------------------------------------------
    "solitary_breaking": {
        "label"       : "Synolakis (1987) — Solitary Wave Run-up (Breaking)",
        "ref"         : "Synolakis, C.E. (1987). J. Fluid Mech. 185, 523–545",
        "g"           : 9.81,
        "d0"          : 1.0,
        "kappa1_d"    : 1.52,
        "R_B"         : 0.0,
        "x_min"       : -100.0,
        "x_max"       : 20.0,
        "t_min"       : 0.0,
        "t_max"       : 10.0,
        "ic_type"     : "solitary",
        "wave_height" : 0.3,
        "wave_center" : -40.0,
        "bottom_type" : "sloping",
        "slope"       : 1.0 / 19.85,
        "slope_start" : -19.85,
        "bc_type"     : "wall_right",
        "N_F"         : 15_000,
        "N_IC"        : 600,
        "N_BC"        : 300,
        "layers"      : [2, 80, 80, 80, 80, 3],
        "activation"  : "tanh",
        "lr_adam"     : 5e-4,
        "epochs_adam" : 10_000,
        "epochs_lbfgs": 3_000,
        "w_pde"       : 1.0,
        "w_ell"       : 1.0,
        "w_ic"        : 10.0,
        "w_bc"        : 5.0,
        "snap_times"  : [3.19, 4.79, 6.39, 7.98, 9.58],
        "plot_freq"   : 500,
    },

    # ------------------------------------------------------------------
    # CASE 4: Beji & Battjes (1993) — §4.5
    # Harmonic wave over submerged trapezoidal bar.
    # Gauges: W4=10.5, W5=12.5, W6=13.5, W7=14.5, W8=15.7, W9=17.3 m
    # ------------------------------------------------------------------
    "beji_battjes": {
        "label"       : "Beji & Battjes (1993) — Harmonic Wave over Submerged Bar",
        "ref"         : "Beji, S. & Battjes, J. (1993). Coast. Eng. 19(1–2), 151–162",
        "g"           : 9.81,
        "d0"          : 0.4,
        "kappa1_d"    : 1.52,
        "R_B"         : 0.0,
        "x_min"       : 0.0,
        "x_max"       : 25.0,
        "t_min"       : 0.0,
        "t_max"       : 40.0,
        "ic_type"     : "still",
        "wave_amp"    : 0.029,
        "wave_period" : 2.525,
        "influx_x"    : 0.0,
        "bottom_type" : "bar",
        "d0_deep"     : 0.4,
        "d0_bar"      : 0.1,
        "bar_x"       : [6.0, 12.0, 14.0, 17.0],
        "bc_type"     : "influx_left_sponge_right",
        "N_F"         : 12_000,
        "N_IC"        : 500,
        "N_BC"        : 400,
        "layers"      : [2, 64, 64, 64, 64, 3],
        "activation"  : "tanh",
        "lr_adam"     : 1e-3,
        "epochs_adam" : 8_000,
        "epochs_lbfgs": 3_000,
        "w_pde"       : 1.0,
        "w_ell"       : 1.0,
        "w_ic"        : 5.0,
        "w_bc"        : 5.0,
        "gauges"      : [10.5, 12.5, 13.5, 14.5, 15.7, 17.3],
        "snap_times"  : [10.0, 20.0, 30.0, 40.0],
        "plot_freq"   : 500,
    },

    # ------------------------------------------------------------------
    # CASE 0: Generic flat-bottom cosine (sanity check)
    # ------------------------------------------------------------------
    "flat_cosine": {
        "label"       : "Generic Flat-bottom Cosine Wave (Sanity Check)",
        "ref"         : "N/A",
        "g"           : 9.81,
        "d0"          : 1.0,
        "kappa1_d"    : 1.52,
        "R_B"         : 0.0,
        "x_min"       : 0.0,
        "x_max"       : 10.0,
        "t_min"       : 0.0,
        "t_max"       : 2.0,
        "ic_type"     : "cosine",
        "wave_amp"    : 0.1,
        "wave_k"      : 2.0,
        "bottom_type" : "flat",
        "bc_type"     : "periodic",
        "N_F"         : 8_000,
        "N_IC"        : 500,
        "N_BC"        : 200,
        "layers"      : [2, 64, 64, 64, 64, 3],
        "activation"  : "tanh",
        "lr_adam"     : 1e-3,
        "epochs_adam" : 5_000,
        "epochs_lbfgs": 2_000,
        "w_pde"       : 1.0,
        "w_ell"       : 1.0,
        "w_ic"        : 10.0,
        "w_bc"        : 5.0,
        "snap_times"  : [0.5, 1.0, 1.5, 2.0],
        "plot_freq"   : 500,
    },
}

print(f"Loaded {len(CASES)} cases. Active: '{ACTIVE_CASE}'")

## 4. VBM Coefficients + Load Active Config

The Variational Boussinesq Model closes the vertical structure with a single profile

$$F(z) = \frac{\cosh\!\big(\kappa (z+d)\big)}{\cosh(\kappa d)} - 1,$$

and the coefficients in eqs. (13)–(15) are its vertical moments:

$$\beta = \int_{-d}^{0}\! F\,dz = \frac{\tanh\kappa d}{\kappa} - d,\qquad
\alpha = \int_{-d}^{0}\! F^2 dz = d\Big(1 + \tfrac12\operatorname{sech}^2\kappa d - \tfrac{3}{2}\tfrac{\tanh \kappa d}{\kappa d}\Big),$$
$$\gamma = \int_{-d}^{0}\! (F')^2 dz = \frac{\kappa}{2}\big(\tanh \kappa d - \kappa d \operatorname{sech}^2 \kappa d\big).$$

Note $\beta < 0$ for all $\kappa d$ — this is a gauge choice (flipping the sign of $F$ flips
the signs of both $\beta$ and $\Psi$); $\beta$ enters the dispersion relation only as $\beta^2$.

The cell below derives $\alpha,\beta,\gamma$ from `kappa1_d` and **verifies** them against the
exact linear dispersion relation $\omega^2 = gk\tanh(kd)$ at $k=\kappa_1$.

In [ ]:
cfg = CASES[ACTIVE_CASE]

G            = cfg["g"]
H0           = cfg["d0"]
KAPPA1_D     = cfg["kappa1_d"]        # κ₁·d₀, held constant across the domain
R_B          = cfg["R_B"]
X_MIN        = cfg["x_min"]
X_MAX        = cfg["x_max"]
T_MIN        = cfg["t_min"]
T_MAX        = cfg["t_max"]
L            = X_MAX - X_MIN
LAYERS       = cfg["layers"]
ACTIVATION   = cfg["activation"]
LR_ADAM      = cfg["lr_adam"]
EPOCHS_ADAM  = cfg["epochs_adam"]
EPOCHS_LBFGS = cfg["epochs_lbfgs"]
N_F          = cfg["N_F"]
N_IC         = cfg["N_IC"]
N_BC         = cfg["N_BC"]
W_PDE        = cfg["w_pde"]
W_ELL        = cfg["w_ell"]
W_IC         = cfg["w_ic"]
W_BC         = cfg["w_bc"]
PLOT_FREQ    = cfg["plot_freq"]
SAVE_DIR     = f"pinn_results/{ACTIVE_CASE}"

# Thin-film floor used for wetting/drying and to keep 1/d finite in γ.
H_MIN = 1e-3 * H0

os.makedirs(SAVE_DIR, exist_ok=True)


# ----------------------------------------------------------------------
#  VBM coefficients — vertical moments of F(z) (see markdown above).
#  With κ₁·d held constant across the domain, the κd-dependent factors are
#  constants and α, β scale linearly with d while γ scales as 1/d.
# ----------------------------------------------------------------------
_C   = KAPPA1_D
_T   = np.tanh(_C)
_S2  = 1.0 / np.cosh(_C) ** 2

A_COEF = 1.0 + 0.5 * _S2 - 1.5 * _T / _C     # α = A_COEF · d
B_COEF = _T / _C - 1.0                       # β = B_COEF · d   (negative)
G_COEF = 0.5 * _C * (_T - _C * _S2)          # γ = G_COEF / d


def vbm_coeffs(d):
    """α, β, γ at local depth d (κ₁·d held fixed at KAPPA1_D)."""
    d = np.asarray(d, dtype=float)
    return A_COEF * d, B_COEF * d, G_COEF / d


ALPHA, BETA, GAMMA = (float(v) for v in vbm_coeffs(H0))   # reference values at d₀

print(f"\n{'='*60}")
print(f"  {cfg['label']}")
print(f"  Ref: {cfg['ref']}")
print(f"{'='*60}")
print(f"  κ₁d={KAPPA1_D}  →  α={ALPHA:.4f}  β={BETA:.4f}  γ={GAMMA:.4f}  (at d₀={H0})")
print(f"  g={G}  R_B={R_B}  h_min={H_MIN:.2e}")
print(f"  Domain  x∈[{X_MIN},{X_MAX}]  t∈[{T_MIN},{T_MAX}]")
print(f"  IC={cfg['ic_type']}  BC={cfg['bc_type']}  Bottom={cfg['bottom_type']}")
print(f"  N_F={N_F}  N_IC={N_IC}  N_BC={N_BC}")
print(f"  Network={LAYERS}  act={ACTIVATION}")
print(f"  Output → {SAVE_DIR}")

# ----------------------------------------------------------------------
#  Dispersion check.  Linearising eqs. (13)–(15) on a flat bottom gives
#      ω² = g k² [ d − β² k² / (α k² + γ) ]
#  which must reproduce the exact ω² = g k tanh(kd) at the design
#  wavenumber k = κ₁.  This is what pins α, β, γ down.
# ----------------------------------------------------------------------
def omega2_vbm(k, d):
    a, b, g_ = vbm_coeffs(d)
    return G * k**2 * (d - b**2 * k**2 / (a * k**2 + g_))


def omega2_exact(k, d):
    return G * k * np.tanh(k * d)


k1 = KAPPA1_D / H0
print(f"\n  Dispersion check at k=κ₁={k1:.4f} rad/m, d={H0} m")
print(f"    exact  ω² = {omega2_exact(k1, H0):.6f}")
print(f"    VBM    ω² = {omega2_vbm(k1, H0):.6f}")
rel = abs(omega2_vbm(k1, H0) / omega2_exact(k1, H0) - 1.0)
print(f"    rel. error = {rel:.2e}")
assert rel < 1e-3, "VBM coefficients do not reproduce exact dispersion at k=κ₁"
print("  ✓ coefficients verified")

## 5. Bottom Depth Profile d(x) — NumPy and differentiable TensorFlow versions

`depth_tf` is a pure-TF function of `x`, so `∂ₓd` is available to the autodiff tape.
This is what makes the shoaling term $u\,\partial_x d$ inside $\partial_x(hu)$ real
rather than silently zero.

`d(x)` is the **true geometry** and goes negative above the still-water shoreline.
The water column is regularised as a thin film,

$$h = \tfrac12\Big(s + \sqrt{s^2 + \varepsilon^2}\Big),\qquad s = d + \eta,\ \varepsilon = h_{\min},$$

which is smooth everywhere, equals $s$ where the water is deep, and decays to $0^+$
rather than going negative on dry bed.

In [ ]:
def bottom_depth(x_arr):
    """True still-water depth d(x) [m] (NumPy).  Negative above the shoreline."""
    x  = np.asarray(x_arr).flatten()
    bt = cfg["bottom_type"]

    if bt == "flat":
        d = np.full_like(x, H0)

    elif bt == "sloping":
        slope       = cfg["slope"]
        slope_start = cfg["slope_start"]
        d = np.where(x < slope_start, H0, H0 - slope * (x - slope_start))

    elif bt == "bar":
        d0_deep = cfg["d0_deep"]
        d0_bar  = cfg["d0_bar"]
        x0, x1, x2, x3 = cfg["bar_x"]
        slope_up   = (d0_deep - d0_bar) / (x1 - x0)
        slope_down = (d0_bar  - d0_deep) / (x3 - x2)
        d = np.where(x < x0,  d0_deep,
            np.where(x < x1,  d0_deep - slope_up   * (x - x0),
            np.where(x < x2,  d0_bar,
            np.where(x < x3,  d0_bar  + slope_down * (x - x2),
                               d0_deep))))
    else:
        d = np.full_like(x, H0)

    return d.reshape(x_arr.shape) if hasattr(x_arr, "shape") else d


# ----------------------------------------------------------------------
#  Differentiable TensorFlow counterparts
# ----------------------------------------------------------------------
def depth_tf(x):
    """d(x) as a pure-TF op — differentiable w.r.t. x via tf.where."""
    bt   = cfg["bottom_type"]
    ones = tf.ones_like(x)

    if bt == "sloping":
        s  = tf.constant(cfg["slope"],       tf.float32)
        x0 = tf.constant(cfg["slope_start"], tf.float32)
        return tf.where(x < x0, H0 * ones, H0 - s * (x - x0))

    if bt == "bar":
        dd, db         = cfg["d0_deep"], cfg["d0_bar"]
        x0, x1, x2, x3 = cfg["bar_x"]
        su = (dd - db) / (x1 - x0)
        sd = (db - dd) / (x3 - x2)
        return tf.where(x < x0, dd * ones,
               tf.where(x < x1, dd - su * (x - x0),
               tf.where(x < x2, db * ones,
               tf.where(x < x3, db + sd * (x - x2), dd * ones))))

    return H0 * ones          # "flat" and fallback


def smooth_pos(s, eps=None):
    """Smooth positive part: → s for s ≫ ε, → 0⁺ for s ≪ −ε.  C^∞ everywhere."""
    e = tf.constant(H_MIN if eps is None else eps, tf.float32)
    return 0.5 * (s + tf.sqrt(s * s + e * e))


def coeffs_tf(d_pos):
    """α(x), β(x), γ(x) from local positive depth, κ₁·d fixed at KAPPA1_D."""
    alpha = tf.constant(A_COEF, tf.float32) * d_pos
    beta  = tf.constant(B_COEF, tf.float32) * d_pos
    gamma = tf.constant(G_COEF, tf.float32) / d_pos
    return alpha, beta, gamma


# ---- consistency check: TF vs NumPy depth, and ∂ₓd against finite differences ----
_xc = tf.constant(np.linspace(X_MIN, X_MAX, 501, dtype=np.float32)[:, None])
with tf.GradientTape() as _tp:
    _tp.watch(_xc)
    _dc = depth_tf(_xc)
_dx = _tp.gradient(_dc, _xc).numpy().flatten()

_dn  = bottom_depth(np.linspace(X_MIN, X_MAX, 501))
_fd  = np.gradient(_dn, np.linspace(X_MIN, X_MAX, 501))
_int = np.abs(_dx[2:-2] - _fd[2:-2])          # skip kinks at the ends
print(f"depth_tf vs bottom_depth : max |Δd|  = {np.abs(_dc.numpy().flatten()-_dn).max():.2e}")
print(f"autodiff ∂ₓd vs finite-diff: median |Δ| = {np.median(_int):.2e}  "
      f"(spikes only at slope breaks)")

# Visualise bottom profile
x_plot = np.linspace(X_MIN, X_MAX, 400)
d_plot = bottom_depth(x_plot)

fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_between(x_plot, -d_plot, -d_plot.max() - 0.1, color="wheat", alpha=0.7, label="bottom")
ax.axhline(0, color="steelblue", lw=1.5, label="still water level")
ax.set(title=f"Bottom Profile — {cfg['label']}",
       xlabel="x [m]", ylabel="z [m]")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "bottom_profile.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"d range: [{d_plot.min():.3f}, {d_plot.max():.3f}] m")
if d_plot.min() <= 0:
    x_sh = x_plot[np.argmax(d_plot <= 0)]
    print(f"Still-water shoreline at x ≈ {x_sh:.2f} m — "
          f"{100*np.mean(d_plot <= 0):.0f}% of the domain is dry at rest "
          f"(handled by the thin-film regularisation).")

## 5b. Exact Carrier–Greenspan Solution

Carrier & Greenspan (1958) solved the **nonlinear** shallow water equations on a plane beach
exactly, via a hodograph transformation. Measuring $X$ offshore from the still-water shoreline
and nondimensionalising with a length $l_0$ ($X = l_0\hat X$, $\eta = s l_0\hat\eta$,
$U = \sqrt{gsl_0}\,\hat U$, $t = \sqrt{l_0/(gs)}\,\hat t$), the system

$$\hat\eta_{\hat t} + [(\hat X + \hat\eta)\hat U]_{\hat X} = 0,\qquad
\hat U_{\hat t} + \hat U\hat U_{\hat X} + \hat\eta_{\hat X} = 0$$

becomes **linear** in the variables $\sigma = 2\sqrt{\hat X + \hat\eta}$ and $\lambda$:

$$\Phi_{\lambda\lambda} = \Phi_{\sigma\sigma} + \frac{1}{\sigma}\Phi_\sigma,
\qquad
\hat U = \frac{\Phi_\sigma}{\sigma},\quad
\hat\eta = -\frac{\Phi_\lambda}{2} - \frac{\hat U^2}{2},\quad
\hat X = \frac{\sigma^2}{4} - \hat\eta,\quad
\hat t = \lambda + \hat U .$$

A monochromatic wave of dimensionless frequency $\hat\omega$ is $\Phi = A J_0(\hat\omega\sigma)\sin(\hat\omega\lambda)$
— the Bessel equation above is satisfied exactly. The map is **implicit**: $(\sigma,\lambda)$
gives $(\hat X,\hat t)$, so evaluating on a physical grid needs a Newton inversion, done
vectorised below.

**Validity.** The map folds — the "solution" becomes multivalued, i.e. the wave breaks — unless

$$A\hat\omega^3 \le 2 \qquad\Longleftrightarrow\qquad R \le \frac{g s^2}{\omega^2},$$

$R$ being the vertical run-up. The cell asserts both this and $\det J > 0$ directly. At the
originally configured $a = 0.03$ m this is violated by a factor 2.6, which is why the case
amplitude is now 0.008 m.

**How it is used.** Two roles, and it is worth being clear that these are different:

1. *Forcing* — $\eta$ and $u$ at $x_{\min}$ (§7, §9) and the initial state (§6). Without this
   the case had no wave-maker at all and simply decayed.
2. *Reference* — the comparison in §17.

The solution is verified below by substituting it back into the nonlinear shallow water
equations with centred finite differences; the residual must vanish to round-off.

In [ ]:
if ACTIVE_CASE == "carrier_greenspan":
    from scipy.special import j0 as _j0, j1 as _j1, jv as _jv

    # ------------------------------------------------------------------
    #  Scales.  X is measured OFFSHORE from the still-water shoreline;
    #  the physical x of the notebook increases SHOREWARD.
    # ------------------------------------------------------------------
    CG_L0 = 1.0                                        # length scale [m]
    CG_S  = cfg["slope"]
    CG_XS = cfg["slope_start"] + cfg["d0"] / CG_S      # still-water shoreline
    CG_T0 = np.sqrt(CG_L0 / (G * CG_S))                # time scale [s]
    CG_W  = 2.0 * np.pi / cfg["wave_period"] * CG_T0   # dimensionless ω
    CG_U0 = np.sqrt(G * CG_S * CG_L0)                  # velocity scale [m/s]

    def _cg_state(sig, lam, A):
        """Hodograph state and its (σ,λ) derivatives for Φ = A J₀(ωσ) sin(ωλ)."""
        z  = CG_W * sig
        S  = np.sin(CG_W * lam)
        C  = np.cos(CG_W * lam)
        zz = np.where(z > 1e-9, z, 1.0)
        gz = np.where(z > 1e-9, _j1(zz) / zz, 0.5)     # J₁(z)/z, limit ½
        gp = np.where(z > 1e-9, -_jv(2, zz) / zz, 0.0) # d/dz[J₁/z] = −J₂/z
        J0z, J1z = _j0(z), _j1(z)

        U   = -A * CG_W**2 * S * gz                     # = Φ_σ/σ
        eta = -0.5 * A * CG_W * J0z * C - 0.5 * U * U   # = −Φ_λ/2 − U²/2

        dU_ds = -A * CG_W**3 * S * gp
        dU_dl = -A * CG_W**3 * C * gz
        de_ds =  0.5 * A * CG_W**2 * J1z * C - U * dU_ds
        de_dl =  0.5 * A * CG_W**2 * J0z * S - U * dU_dl
        return U, eta, dU_ds, dU_dl, de_ds, de_dl

    def _cg_invert(Xh, th, A, iters=80):
        """
        (X̂, t̂) → (σ, λ) by damped Newton, vectorised.

        Points on dry bed have no solution at all, so the iteration is expected
        to diverge there; errstate keeps that from spamming warnings, the
        isfinite guards keep it from poisoning neighbouring entries, and the
        final residual test is what actually decides convergence.
        """
        with np.errstate(all="ignore"):
            sig = 2.0 * np.sqrt(np.maximum(Xh, 1e-12))
            lam = np.array(th, dtype=float, copy=True)
            for _ in range(iters):
                U, eta, dU_ds, dU_dl, de_ds, de_dl = _cg_state(sig, lam, A)
                f1 = sig * sig / 4.0 - eta - Xh
                f2 = lam + U - th
                j11, j12 = sig / 2.0 - de_ds, -de_dl
                j21, j22 = dU_ds, 1.0 + dU_dl
                det  = j11 * j22 - j12 * j21
                bad  = ~np.isfinite(det) | (np.abs(det) < 1e-14)
                detc = np.where(bad, 1.0, det)
                ds = np.where(bad, 0.0, (-f1 * j22 + f2 * j12) / detc)
                dl = np.where(bad, 0.0, (-f2 * j11 + f1 * j21) / detc)
                m  = np.maximum(np.abs(ds), np.abs(dl))
                f  = np.where(m < 0.5, 1.0, 0.5 / np.maximum(m, 1e-30))
                sig = np.maximum(0.0, sig + f * ds)
                lam = lam + f * dl
                sig = np.where(np.isfinite(sig), sig, 0.0)
                lam = np.where(np.isfinite(lam), lam, th)
            U, eta, *_ = _cg_state(sig, lam, A)
            res = np.maximum(np.abs(sig * sig / 4.0 - eta - Xh),
                             np.abs(lam + U - th))
            ok = np.isfinite(res) & (res < 1e-9) & (sig > 1e-7)
        return sig, lam, ok

    CG_A = 1.0     # provisional; calibrated below

    def cg_exact(x, t, A=None):
        """
        Exact Carrier–Greenspan (η [m], u [m/s]) at physical (x, t).
        u is positive SHOREWARD, matching the +x convention of the notebook.
        Returns NaN on dry bed, where the solution is undefined.
        """
        A = CG_A if A is None else A
        x, t = np.broadcast_arrays(np.asarray(x, float), np.asarray(t, float))
        sig, lam, ok = _cg_invert((CG_XS - x) / CG_L0, t / CG_T0, A)
        with np.errstate(all="ignore"):
            U, eta, *_ = _cg_state(sig, lam, A)
        return (np.where(ok, CG_S * CG_L0 * eta, np.nan),
                np.where(ok, -CG_U0 * U,         np.nan))

    # ------------------------------------------------------------------
    #  Calibrate A so the η amplitude at the offshore boundary is wave_amp
    # ------------------------------------------------------------------
    _tc = np.linspace(0.0, cfg["wave_period"], 241)
    _xc = np.full_like(_tc, X_MIN)
    for _ in range(50):
        _e = cg_exact(_xc, _tc, CG_A)[0]
        _e = _e[np.isfinite(_e)]
        if _e.size == 0:
            raise RuntimeError("CG calibration failed — no wet points at x_min")
        _amp = 0.5 * (_e.max() - _e.min())
        CG_A *= cfg["wave_amp"] / _amp
        if abs(_amp - cfg["wave_amp"]) < 1e-12:
            break

    # ------------------------------------------------------------------
    #  Breaking check.  The hodograph map folds (the "solution" becomes
    #  multivalued) unless A·ω̂³ ≤ 2, equivalently R ≤ g·s²/ω².
    # ------------------------------------------------------------------
    CG_BREAK = CG_A * CG_W**3
    _sg = np.linspace(1e-3, 2.0 * np.sqrt((CG_XS - X_MIN) / CG_L0), 200)
    _lm = np.linspace(0.0, 2.0 * np.pi / CG_W, 200)
    _SG, _LM = np.meshgrid(_sg, _lm)
    _U, _E, _dUs, _dUl, _des, _delam = _cg_state(_SG, _LM, CG_A)
    _detJ = (_SG / 2.0 - _des) * (1.0 + _dUl) - (-_delam) * _dUs
    _det_min = float(_detJ.min())

    _lam_s = np.linspace(0.0, 2.0 * np.pi / CG_W, 2001)
    _R = float(np.max(CG_S * CG_L0 * _cg_state(np.zeros_like(_lam_s), _lam_s, CG_A)[1]))

    print("Carrier–Greenspan exact solution")
    print(f"  shoreline (still water) x_s = {CG_XS:.3f} m   slope s = {CG_S:.4f}")
    print(f"  T0 = {CG_T0:.4f} s   omega_hat = {CG_W:.4f}   A = {CG_A:.6f}")
    print(f"  offshore amplitude at x_min = {cfg['wave_amp']:.5f} m")
    print(f"  vertical run-up R = {_R:.4f} m  →  shoreline sweeps "
          f"[{CG_XS - _R/CG_S:.2f}, {CG_XS + _R/CG_S:.2f}] m   (x_max = {X_MAX})")
    print(f"  breaking parameter A·omega_hat^3 = {CG_BREAK:.4f}  (limit 2)")
    print(f"  min det J = {_det_min:.5f}  "
          f"({'single-valued ✓' if _det_min > 0 else 'MULTIVALUED ✗'})")

    assert CG_BREAK <= 2.0, (
        f"wave_amp={cfg['wave_amp']} m breaks: A·omega_hat^3={CG_BREAK:.3f} > 2. "
        f"Max admissible amplitude for this slope/period is about "
        f"{cfg['wave_amp'] * 2.0 / CG_BREAK:.5f} m.")
    assert _det_min > 0, "hodograph map folds — CG solution is multivalued"
    assert CG_XS + _R / CG_S < X_MAX, "run-up exceeds x_max — extend the domain"

    # ------------------------------------------------------------------
    #  Verification: does it actually solve the nonlinear shallow water
    #  equations?  Centred finite differences in the always-wet region.
    #  This validates the hodograph algebra, the signs and the scaling.
    # ------------------------------------------------------------------
    _h  = 1e-4
    _xv = np.linspace(X_MIN + 0.3, CG_XS - 1.0, 15)
    _tv = np.linspace(1.0, cfg["wave_period"] + 1.0, 12)
    _XV, _TV = np.meshgrid(_xv, _tv)
    _e0, _u0 = cg_exact(_XV, _TV)
    _ep, _up = cg_exact(_XV + _h, _TV)
    _em, _um = cg_exact(_XV - _h, _TV)
    _et, _ut = cg_exact(_XV, _TV + _h)
    _eb, _ub = cg_exact(_XV, _TV - _h)

    _eta_t = (_et - _eb) / (2 * _h)
    _u_t   = (_ut - _ub) / (2 * _h)
    _flux  = ((bottom_depth(_XV + _h) + _ep) * _up
            - (bottom_depth(_XV - _h) + _em) * _um) / (2 * _h)
    _mass  = _eta_t + _flux
    _mom   = _u_t + _u0 * (_up - _um) / (2 * _h) + G * (_ep - _em) / (2 * _h)

    _rm = np.nanmax(np.abs(_mass)) / np.nanmax(np.abs(_eta_t))
    _rp = np.nanmax(np.abs(_mom))  / np.nanmax(np.abs(_u_t))
    print(f"  NSWE residual (relative): mass {_rm:.2e}   momentum {_rp:.2e}")
    assert _rm < 1e-5 and _rp < 1e-5, "CG solution does not satisfy the NSWE"
    print("  ✓ exact solution verified against the nonlinear shallow water equations")

    # ------------------------------------------------------------------
    #  Picture of the reference solution
    # ------------------------------------------------------------------
    _xp = np.linspace(X_MIN, X_MAX, 400)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    for _f, _c in zip(np.linspace(0, 1, 5)[:-1], plt.cm.viridis(np.linspace(0, 1, 4))):
        _tt = _f * cfg["wave_period"]
        _ee, _ = cg_exact(_xp, np.full_like(_xp, _tt))
        axes[0].plot(_xp, _ee, color=_c, lw=1.4, label=f"t = {_tt:.1f} s")
    axes[0].plot(_xp, -bottom_depth(_xp), color="saddlebrown", lw=1.2, label="bed")
    axes[0].axvline(CG_XS, color="crimson", ls=":", lw=1)
    axes[0].set(title="Exact CG surface elevation over one period",
                xlabel="x [m]", ylabel=r"$\eta$ [m]")
    axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

    # The shoreline (σ=0) is naturally parametrised by λ, but the physical time
    # is t = (λ + Û)·T0, not λ·T0 — a ~1 s offset here, so use the exact map.
    _lam_p = np.linspace(0.0, T_MAX / CG_T0 + 2.0, 3000)
    _Us, _Es, *_ = _cg_state(np.zeros_like(_lam_p), _lam_p, CG_A)
    _ts_sh = (_lam_p + _Us) * CG_T0
    _xs_sh = CG_XS + CG_L0 * _Es
    _keep  = (_ts_sh >= T_MIN) & (_ts_sh <= T_MAX)
    axes[1].plot(_ts_sh[_keep], _xs_sh[_keep], color="steelblue", lw=1.2)
    axes[1].axhline(CG_XS, color="crimson", ls=":", lw=1, label="still-water shoreline")
    axes[1].set(title=r"Exact shoreline position $x_s(t)$  ($\sigma=0$)",
                xlabel="t [s]", ylabel="x [m]")
    axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "cg_exact.png"), dpi=150, bbox_inches="tight")
    plt.show()

else:
    print(f"Case '{ACTIVE_CASE}' has no closed-form reference — section skipped.")

## 6. Initial Conditions

In [ ]:
def initial_condition(x_arr):
    """
    Returns (η₀, u₀, Ψ₀) at t=0.

    Ψ₀ is returned as zeros for signature compatibility only — it is NOT
    imposed. Ψ has no independent initial condition: eq. (15) is elliptic
    (instantaneous), so Ψ is slaved to u at every instant, including t=0.
    Forcing Ψ(x,0)=0 alongside a non-zero u₀ over-determines the problem.
    """
    x  = np.asarray(x_arr)
    ic = cfg["ic_type"]

    if ic == "cosine":
        eta0 = cfg["wave_amp"] * np.cos(2 * np.pi * cfg["wave_k"] * x / L)
        u0   = np.zeros_like(x)

    elif ic == "solitary":
        H   = cfg["wave_height"]
        x0  = cfg["wave_center"]
        arg = np.sqrt(3.0 * H / (4.0 * H0**3)) * (x - x0)
        eta0 = H / np.cosh(arg)**2
        u0   = np.sqrt(G / H0) * eta0 / (1.0 + eta0 / H0)

    elif ic == "carrier_greenspan":
        # Exact CG state at t=0.  At t=0 we have λ≈0 ⇒ u≡0 everywhere and the
        # shoreline sits at maximum run-down, so this is a clean initial state.
        eta0, u0 = cg_exact(x, np.zeros_like(x))
        dry  = ~np.isfinite(eta0)
        d_x  = bottom_depth(x)
        eta0 = np.where(dry, -d_x, eta0)      # dry bed ⇒ h = d + η = 0
        u0   = np.where(dry, 0.0,  u0)

    elif ic == "harmonic_seed":
        amp   = cfg["wave_amp"] * 0.01
        k_w   = 2 * np.pi / (cfg["wave_period"] * np.sqrt(G * H0))
        eta0  = amp * np.cos(k_w * (x - cfg["influx_x"]))
        u0    = np.zeros_like(x)

    else:  # "still"
        eta0 = np.zeros_like(x)
        u0   = np.zeros_like(x)

    psi0 = np.zeros_like(x)
    return eta0, u0, psi0


# Visualise IC
eta0_plot, u0_plot, _ = initial_condition(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(x_plot, eta0_plot, color="steelblue", label=r"$\eta(x,0)$")
axes[0].plot(x_plot, -d_plot, color="saddlebrown", lw=1, ls="--", label="bed $-d(x)$")
axes[0].set(title=r"Initial $\eta(x,0)$", xlabel="x [m]", ylabel=r"$\eta$ [m]")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)
axes[1].plot(x_plot, u0_plot, color="darkorange")
axes[1].set(title=r"Initial $u(x,0)$", xlabel="x [m]", ylabel="u [m/s]")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "initial_condition.png"), dpi=150, bbox_inches="tight")
plt.show()

h0_plot = d_plot + eta0_plot
print(f"η₀ range: [{eta0_plot.min():.4f}, {eta0_plot.max():.4f}] m")
print(f"u₀ range: [{u0_plot.min():.4f}, {u0_plot.max():.4f}] m/s")
print(f"h₀ = d+η₀ min: {h0_plot.min():.2e} m  (must be ≥ 0)")
assert np.all(np.isfinite(eta0_plot)) and np.all(np.isfinite(u0_plot)), "non-finite IC"
assert h0_plot.min() > -1e-12, "initial water column is negative somewhere"

## 7. Collocation Point Sampling

In [ ]:
def sample_points():
    x_f  = np.random.uniform(X_MIN, X_MAX, (N_F,  1)).astype(np.float32)
    t_f  = np.random.uniform(T_MIN, T_MAX, (N_F,  1)).astype(np.float32)
    x_ic = np.random.uniform(X_MIN, X_MAX, (N_IC, 1)).astype(np.float32)
    t_ic = np.zeros((N_IC, 1), dtype=np.float32)
    eta_ic, u_ic, psi_ic = initial_condition(x_ic)
    t_bc       = np.random.uniform(T_MIN, T_MAX, (N_BC, 1)).astype(np.float32)
    x_bc_left  = np.full((N_BC, 1), X_MIN, dtype=np.float32)
    x_bc_right = np.full((N_BC, 1), X_MAX, dtype=np.float32)

    # Offshore forcing.  For Carrier–Greenspan the incident wave is imposed by
    # evaluating the exact solution at x_min — this is what actually drives the
    # problem.  (Previously the case had no influx at all: bc_type was
    # "wall_right" only and cfg["influx_x"] was never used, so nothing forced
    # the domain and the solution simply decayed to rest.)
    if cfg["bc_type"] == "cg_influx":
        eta_bc, u_bc = cg_exact(x_bc_left, t_bc)
        assert np.all(np.isfinite(eta_bc)), "CG solution undefined at x_min"
        eta_bc = eta_bc.astype(np.float32)
        u_bc   = u_bc.astype(np.float32)
    else:
        eta_bc = np.zeros((N_BC, 1), dtype=np.float32)
        u_bc   = np.zeros((N_BC, 1), dtype=np.float32)

    return {
        "x_f"        : tf.constant(x_f),
        "t_f"        : tf.constant(t_f),
        "x_ic"       : tf.constant(x_ic),
        "t_ic"       : tf.constant(t_ic),
        "eta_ic"     : tf.constant(eta_ic.astype(np.float32)),
        "u_ic"       : tf.constant(u_ic.astype(np.float32)),
        "psi_ic"     : tf.constant(psi_ic.astype(np.float32)),
        "t_bc"       : tf.constant(t_bc),
        "x_bc_left"  : tf.constant(x_bc_left),
        "x_bc_right" : tf.constant(x_bc_right),
        "eta_bc"     : tf.constant(eta_bc),
        "u_bc"       : tf.constant(u_bc),
    }


data = sample_points()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2),
                         gridspec_kw={"width_ratios": [2, 1]})
axes[0].scatter(data["x_f"].numpy(), data["t_f"].numpy(),
                s=0.3, alpha=0.3, color="steelblue", label=f"PDE ({N_F})")
axes[0].scatter(data["x_ic"].numpy(), data["t_ic"].numpy(),
                s=2, alpha=0.6, color="green", label=f"IC ({N_IC})")
axes[0].scatter(data["x_bc_left"].numpy(), data["t_bc"].numpy(),
                s=2, alpha=0.6, color="red", label=f"BC ({N_BC})")
axes[0].scatter(data["x_bc_right"].numpy(), data["t_bc"].numpy(),
                s=2, alpha=0.6, color="red")
axes[0].set(title="Collocation Points", xlabel="x [m]", ylabel="t [s]")
axes[0].legend(markerscale=5, fontsize=7)
axes[0].grid(True, alpha=0.3)

if cfg["bc_type"] == "cg_influx":
    order = np.argsort(data["t_bc"].numpy().flatten())
    axes[1].plot(data["t_bc"].numpy().flatten()[order],
                 data["eta_bc"].numpy().flatten()[order], lw=0.8, color="steelblue",
                 label=r"$\eta(x_{\min},t)$")
    axes[1].plot(data["t_bc"].numpy().flatten()[order],
                 data["u_bc"].numpy().flatten()[order], lw=0.8, color="darkorange",
                 label=r"$u(x_{\min},t)$")
    axes[1].set(title="Offshore forcing (exact CG)", xlabel="t [s]")
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].axis("off")
    axes[1].text(0.5, 0.5, f"bc_type = {cfg['bc_type']}", ha="center", va="center")

plt.tight_layout()
plt.show()
print(f"PDE: {N_F}  IC: {N_IC}  BC: {N_BC}")

## 8. Neural Network

In [ ]:
class PINN(tf.keras.Model):
    """Fully-connected network: (x, t) → (η, u, Ψ)"""

    def __init__(self, layers_cfg=LAYERS, activation=ACTIVATION):
        super().__init__()
        self.x_lb = tf.constant([[X_MIN, T_MIN]], dtype=tf.float32)
        self.x_ub = tf.constant([[X_MAX, T_MAX]], dtype=tf.float32)
        self.hidden = [
            tf.keras.layers.Dense(units, activation=activation,
                                  kernel_initializer="glorot_normal")
            for units in layers_cfg[1:-1]
        ]
        self.out_layer = tf.keras.layers.Dense(
            layers_cfg[-1], activation=None, kernel_initializer="glorot_normal"
        )

    def normalize(self, x, t):
        xt = tf.concat([x, t], axis=1)
        return 2.0 * (xt - self.x_lb) / (self.x_ub - self.x_lb) - 1.0

    def call(self, x, t):
        h = self.normalize(x, t)
        for layer in self.hidden:
            h = layer(h)
        out = self.out_layer(h)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]


model = PINN(layers_cfg=LAYERS, activation=ACTIVATION)
_ = model(data["x_ic"], data["t_ic"])   # build
model.summary()

## 9. Physics Residuals & Loss

Residuals in **divergence form**, so the $x$-dependence of $\alpha,\beta$ is differentiated
rather than treated as constant:

$$r_\eta = \partial_t \eta + \partial_x(hu) + \partial_x\!\big(\beta\,\partial_x\Psi\big)$$
$$r_u = \partial_t u + g\,\partial_x \eta + u\,\partial_x u + R_B$$
$$r_\Psi = -\partial_x\!\big(\alpha\,\partial_x\Psi\big) + \gamma\Psi - \partial_x(\beta u)$$

with $h = \text{smooth}^+(d(x)+\eta)$ and $\alpha,\beta,\gamma$ evaluated at the local depth.
Because `depth_tf` is differentiable, $\partial_x(hu)$ now carries the shoaling term
$u\,\partial_x d$ — the term that was missing when $h = H_0 + \eta$ was hardcoded.

> $R_B$ comes from the config and is `0.0` for every case: **no breaker model is
> implemented**, so `solitary_breaking` currently runs as a non-breaking simulation.

In [ ]:
@tf.function
def compute_residuals(model, x, t):
    """PDE residuals for the weakly nonlinear VBM (eqs. 13–15), divergence form."""
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([x, t])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([x, t])
            eta, u, psi = model(x, t)

            d     = depth_tf(x)                 # differentiable in x
            d_pos = smooth_pos(d) + H_MIN       # strictly positive, for α, β, γ
            alpha, beta, gamma = coeffs_tf(d_pos)

            h  = smooth_pos(d + eta)            # thin-film water column
            hu = h * u

        eta_t = tape1.gradient(eta, t)
        eta_x = tape1.gradient(eta, x)
        u_t   = tape1.gradient(u,   t)
        u_x   = tape1.gradient(u,   x)
        psi_x = tape1.gradient(psi, x)
        hu_x  = tape1.gradient(hu,  x)          # includes u·∂ₓd

        # Fluxes whose x-derivative is taken by the outer tape
        flux_b_psi = beta  * psi_x
        flux_a_psi = alpha * psi_x
        flux_b_u   = beta  * u

    d_flux_b_psi = tape2.gradient(flux_b_psi, x)
    d_flux_a_psi = tape2.gradient(flux_a_psi, x)
    d_flux_b_u   = tape2.gradient(flux_b_u,   x)

    # Eq. (13): ∂_t η + ∂_x(hu) + ∂_x(β ∂_x Ψ) = 0
    res_eta = eta_t + hu_x + d_flux_b_psi
    # Eq. (14): ∂_t u + g ∂_x η + u ∂_x u + R_B = 0
    res_u   = u_t + G * eta_x + u * u_x + R_B
    # Eq. (15): -∂_x(α ∂_x Ψ) + γ Ψ - ∂_x(β u) = 0
    res_psi = -d_flux_a_psi + gamma * psi - d_flux_b_u

    return res_eta, res_u, res_psi


@tf.function
def compute_loss(model, data):
    r_eta, r_u, r_psi = compute_residuals(model, data["x_f"], data["t_f"])
    loss_pde = tf.reduce_mean(tf.square(r_eta)) + tf.reduce_mean(tf.square(r_u))
    loss_ell = tf.reduce_mean(tf.square(r_psi))

    # Ψ is deliberately excluded: eq. (15) is elliptic, so Ψ is slaved to u at
    # every instant and has no independent initial condition.  Imposing
    # Ψ(x,0)=0 next to a non-zero u₀ over-determines the problem.
    eta_p, u_p, _ = model(data["x_ic"], data["t_ic"])
    loss_ic = (tf.reduce_mean(tf.square(eta_p - data["eta_ic"]))
             + tf.reduce_mean(tf.square(u_p   - data["u_ic"])))

    bc_type = cfg["bc_type"]
    if bc_type == "periodic":
        eta_l, u_l, psi_l = model(data["x_bc_left"],  data["t_bc"])
        eta_r, u_r, psi_r = model(data["x_bc_right"], data["t_bc"])
        loss_bc = (tf.reduce_mean(tf.square(eta_l - eta_r))
                 + tf.reduce_mean(tf.square(u_l   - u_r))
                 + tf.reduce_mean(tf.square(psi_l - psi_r)))
    elif bc_type == "wall_right":
        _, u_r, _ = model(data["x_bc_right"], data["t_bc"])
        loss_bc = tf.reduce_mean(tf.square(u_r))
    elif bc_type == "cg_influx":
        # Offshore: impose the exact incident state.  Both η and u are given,
        # which over-specifies the single incoming characteristic — harmless
        # here because the imposed pair is an exact solution of the system.
        # Shoreward: dry bed at x_max, no flux.
        eta_l, u_l, _ = model(data["x_bc_left"], data["t_bc"])
        _,     u_r, _ = model(data["x_bc_right"], data["t_bc"])
        loss_bc = (tf.reduce_mean(tf.square(eta_l - data["eta_bc"]))
                 + tf.reduce_mean(tf.square(u_l   - data["u_bc"]))
                 + tf.reduce_mean(tf.square(u_r)))
    elif bc_type == "influx_left_sponge_right":
        amp    = tf.constant(cfg["wave_amp"],    dtype=tf.float32)
        T_wave = tf.constant(cfg["wave_period"], dtype=tf.float32)
        eta_influx = amp * tf.cos(2.0 * np.pi / T_wave * data["t_bc"])
        eta_l, _, _ = model(data["x_bc_left"], data["t_bc"])
        loss_bc = tf.reduce_mean(tf.square(eta_l - eta_influx))
    else:
        loss_bc = tf.constant(0.0)

    total = W_PDE*loss_pde + W_ELL*loss_ell + W_IC*loss_ic + W_BC*loss_bc
    return total, loss_pde, loss_ell, loss_ic, loss_bc


# ---- smoke test: residuals must be finite and the bathymetry must matter ----
_r = compute_residuals(model, data["x_f"][:256], data["t_f"][:256])
assert all(np.all(np.isfinite(v.numpy())) for v in _r), "non-finite residual"
print("Residual functions compiled ✓")
print(f"  |r_η| mean = {np.abs(_r[0].numpy()).mean():.3e}")
print(f"  |r_u| mean = {np.abs(_r[1].numpy()).mean():.3e}")
print(f"  |r_Ψ| mean = {np.abs(_r[2].numpy()).mean():.3e}")

_l = compute_loss(model, data)
assert np.all(np.isfinite([float(v) for v in _l])), "non-finite loss"
print(f"  initial loss: total={float(_l[0]):.3e}  pde={float(_l[1]):.3e}  "
      f"ell={float(_l[2]):.3e}  ic={float(_l[3]):.3e}  bc={float(_l[4]):.3e}")
assert float(_l[4]) > 0 or cfg["bc_type"] == "none", \
    "BC loss is exactly zero — the domain is not being forced"

# ∂ₓd must be non-zero somewhere, or we are back to the flat-bottom bug
_xs = tf.constant(np.linspace(X_MIN, X_MAX, 512, dtype=np.float32)[:, None])
with tf.GradientTape() as _g:
    _g.watch(_xs)
    _dd = depth_tf(_xs)
_slope_max = np.abs(_g.gradient(_dd, _xs).numpy()).max()
print(f"  max |∂ₓd| = {_slope_max:.4f}"
      + ("  ✓ bathymetry is active in the PDE"
         if _slope_max > 0 or cfg["bottom_type"] == "flat"
         else "  ✗ bathymetry is flat — check depth_tf"))

## 10. Training — Phase 1: Adam

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LR_ADAM)
history   = {"total": [], "pde": [], "ell": [], "ic": [], "bc": []}

@tf.function
def train_step():
    with tf.GradientTape() as tape:
        total, l_pde, l_ell, l_ic, l_bc = compute_loss(model, data)
    grads = tape.gradient(total, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total, l_pde, l_ell, l_ic, l_bc


print(f"Adam — {EPOCHS_ADAM} epochs\n")
t0 = time.time()

for epoch in range(1, EPOCHS_ADAM + 1):
    total, l_pde, l_ell, l_ic, l_bc = train_step()
    for k, v in zip(["total","pde","ell","ic","bc"],
                    [total, l_pde, l_ell, l_ic, l_bc]):
        history[k].append(float(v))
    if epoch % PLOT_FREQ == 0 or epoch == 1:
        print(f"  {epoch:>6d}/{EPOCHS_ADAM} | "
              f"Total: {total:.3e} | PDE: {l_pde:.3e} | "
              f"Ell: {l_ell:.3e} | IC: {l_ic:.3e} | "
              f"BC: {l_bc:.3e} | {time.time()-t0:.1f}s")

print(f"\nAdam done in {time.time()-t0:.1f}s")

## 11. Training — Phase 2: L-BFGS-B

In [ ]:
from scipy.optimize import minimize

def loss_and_grad_np(w_flat):
    idx, new_w = 0, []
    for w in model.trainable_variables:
        sz = tf.size(w).numpy()
        new_w.append(w_flat[idx:idx+sz].reshape(w.shape))
        idx += sz
    model.set_weights(new_w)
    with tf.GradientTape() as tape:
        total, l_pde, l_ell, l_ic, l_bc = compute_loss(model, data)
    grads = tape.gradient(total, model.trainable_variables)
    for k, v in zip(["total","pde","ell","ic","bc"],
                    [total, l_pde, l_ell, l_ic, l_bc]):
        history[k].append(float(v))
    g_flat = np.concatenate([g.numpy().flatten() for g in grads]).astype(np.float64)
    return float(total), g_flat


w0 = np.concatenate(
    [w.numpy().flatten() for w in model.trainable_variables]
).astype(np.float64)

print(f"L-BFGS-B — up to {EPOCHS_LBFGS} iterations …")
t1 = time.time()

result = minimize(
    loss_and_grad_np, w0, method="L-BFGS-B", jac=True,
    options={"maxiter": EPOCHS_LBFGS, "ftol": 1e-12, "gtol": 1e-8}
)
print(f"{result.message}")
print(f"Final loss: {result.fun:.3e}  |  Time: {time.time()-t1:.1f}s")

## 12. Loss Curves

In [ ]:
epochs   = np.arange(1, len(history["total"]) + 1)
adam_end = EPOCHS_ADAM

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogy(epochs, history["total"], "k", lw=1.5)
axes[0].axvline(adam_end, color="gray", ls="--", label="Adam → L-BFGS")
axes[0].set(title="Total Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

for key, lbl, col in [("pde","PDE (η,u)","steelblue"),
                       ("ell","Elliptic (Ψ)","darkorange"),
                       ("ic", "IC","green"),
                       ("bc", "BC","red")]:
    axes[1].semilogy(epochs, history[key], label=lbl, color=col, lw=1.2)
axes[1].axvline(adam_end, color="gray", ls="--")
axes[1].set(title="Loss Components", xlabel="Epoch", ylabel="Loss")
axes[1].legend()
axes[1].grid(True, which="both", alpha=0.3)

plt.suptitle(f"Training — {cfg['label']}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "loss_history.png"), dpi=150, bbox_inches="tight")
plt.show()

## 13. Prediction on Grid

In [ ]:
NX, NT = 200, 100
x_vals = np.linspace(X_MIN, X_MAX, NX, dtype=np.float32)
t_vals = np.linspace(T_MIN, T_MAX, NT, dtype=np.float32)
XX, TT = np.meshgrid(x_vals, t_vals)

eta_p, u_p, psi_p = model(
    tf.constant(XX.flatten()[:, None]),
    tf.constant(TT.flatten()[:, None])
)
ETA = eta_p.numpy().reshape(NT, NX)
U   = u_p.numpy().reshape(NT, NX)
PSI = psi_p.numpy().reshape(NT, NX)

print(f"η  range: [{ETA.min():.4f}, {ETA.max():.4f}] m")
print(f"u  range: [{U.min():.4f}, {U.max():.4f}] m/s")
print(f"Ψ  range: [{PSI.min():.4f}, {PSI.max():.4f}]")

## 14. Space-Time Heatmaps

In [ ]:
d_vals = bottom_depth(x_vals)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, F, title, cmap in zip(
    axes,
    [ETA, U, PSI],
    [r"$\eta(x,t)$ [m]", r"$u(x,t)$ [m/s]", r"$\Psi(x,t)$"],
    ["RdBu_r", "RdBu_r", "PuOr_r"]
):
    vmax = np.abs(F).max() or 1.0
    im = ax.pcolormesh(x_vals, t_vals, F,
                       cmap=cmap, vmin=-vmax, vmax=vmax, shading="auto")
    plt.colorbar(im, ax=ax)
    ax.set(xlabel="x [m]", ylabel="t [s]", title=title)

# Bottom profile overlay on η panel
axes[0].plot(x_vals, -d_vals / d_vals.max() * T_MAX * 0.1,
             "k--", lw=1, alpha=0.4, label="−d(x) (scaled)")
axes[0].legend(fontsize=7)

plt.suptitle(f"PINN Solution — {cfg['label']}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fields_spacetime.png"), dpi=150, bbox_inches="tight")
plt.show()

## 15. Snapshots at Selected Times

In [ ]:
snap_times = cfg.get("snap_times", [T_MIN, T_MAX])
colors     = plt.cm.viridis(np.linspace(0, 1, len(snap_times)))
HAS_EXACT  = (ACTIVE_CASE == "carrier_greenspan")

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for i, t_snap in enumerate(snap_times):
    idx      = np.argmin(np.abs(t_vals - t_snap))
    t_actual = t_vals[idx]
    axes[0].plot(x_vals, ETA[idx], color=colors[i], label=f"t = {t_actual:.2f} s")
    axes[1].plot(x_vals, U[idx],   color=colors[i], label=f"t = {t_actual:.2f} s")

    if HAS_EXACT:
        # Exact Carrier–Greenspan (dashed).  NaN on dry bed, so it simply
        # stops at the instantaneous shoreline.
        eta_e, u_e = cg_exact(x_vals, np.full_like(x_vals, t_actual))
        axes[0].plot(x_vals, eta_e, "--", color=colors[i], alpha=0.9, lw=1.2)
        axes[1].plot(x_vals, u_e,   "--", color=colors[i], alpha=0.9, lw=1.2)

# Bottom shading
axes[0].fill_between(x_vals, -d_vals, -d_vals.max() - 0.05,
                     color="wheat", alpha=0.4, label="bottom")
if HAS_EXACT:
    axes[0].axvline(CG_XS, color="crimson", ls=":", lw=1,
                    label=f"still-water shoreline ({CG_XS:.1f} m)")

title0 = (r"Surface elevation $\eta(x,t)$  — solid: PINN, dashed: exact Carrier–Greenspan"
          if HAS_EXACT else r"Surface elevation $\eta(x,t)$")
axes[0].set(title=title0, ylabel=r"$\eta$ [m]")
axes[0].legend(fontsize=7, loc="upper right", ncol=2)
axes[0].grid(True, alpha=0.3)
axes[1].set(title=r"Depth-averaged velocity $u(x,t)$"
                  + ("  — solid: PINN, dashed: exact" if HAS_EXACT else ""),
            xlabel="x [m]", ylabel="u [m/s]")
axes[1].legend(fontsize=7, loc="upper right", ncol=2)
axes[1].grid(True, alpha=0.3)

plt.suptitle(cfg["label"], y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "snapshots.png"), dpi=150, bbox_inches="tight")
plt.show()

## 16. Wave Gauge Signals (Beji-Battjes only)

In [ ]:
if "gauges" in cfg:
    gauges = cfg["gauges"]
    t_g    = np.linspace(T_MIN, T_MAX, 400, dtype=np.float32)
    colors_g = plt.cm.tab10(np.linspace(0, 1, len(gauges)))

    fig, axes = plt.subplots(len(gauges), 1,
                             figsize=(10, 2 * len(gauges)), sharex=True)
    if len(gauges) == 1:
        axes = [axes]

    for ax, xg, col in zip(axes, gauges, colors_g):
        x_g_arr = np.full((len(t_g), 1), xg, dtype=np.float32)
        eta_g, _, _ = model(tf.constant(x_g_arr), tf.constant(t_g[:, None]))
        ax.plot(t_g, eta_g.numpy().flatten(), color=col)
        ax.set_ylabel(r"$\eta$ [m]", fontsize=8)
        ax.set_title(f"Gauge x = {xg} m", fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("t [s]")
    plt.suptitle(f"Wave Gauge Signals — {cfg['label']}", y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "gauge_signals.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No gauges defined for this case — skipping.")

## 17. Metrics vs the Exact Carrier–Greenspan Solution

Compared against the **exact nonlinear** CG solution (§5b), not a linear surrogate.

Reported per snapshot and globally over the wet part of the space–time domain:

- **RMSE** of $\eta$ and $u$, and RMSE normalised by the offshore wave amplitude
- **Pearson correlation** (not cosine similarity — for a zero-mean oscillation the two
  nearly coincide, but Pearson is the defensible one to quote)
- **Maximum run-up**, the headline quantity in run-up benchmarks

> **Expected accuracy floor.** CG is non-dispersive and the VBM is not, so even a perfectly
> converged PINN cannot beat the model gap of $O((kd)^2/3)\approx 0.7\%$ here. Treat ~1%
> normalised RMSE as agreement; anything much larger is a PINN convergence problem, not physics.
>
> Points on dry bed are excluded — the exact solution is undefined there.

In [ ]:
if ACTIVE_CASE == "carrier_greenspan":
    A_REF = cfg["wave_amp"]          # normalisation scale

    # ---- exact solution on the full evaluation grid --------------------
    XXe, TTe = np.meshgrid(x_vals, t_vals)
    ETA_E, U_E = cg_exact(XXe, TTe)
    wet = np.isfinite(ETA_E)
    print(f"Wet fraction of the space-time grid: {100*wet.mean():.1f}% "
          f"({wet.sum()} of {wet.size} points)")

    def _stats(pred, exact, mask):
        p, e = pred[mask], exact[mask]
        rmse = np.sqrt(np.mean((p - e)**2))
        corr = np.corrcoef(p, e)[0, 1] if p.size > 2 else np.nan
        return rmse, corr

    # ---- per-snapshot table -------------------------------------------
    rows = []
    hdr = (f"{'t [s]':>8}  {'RMSE eta':>11}  {'/amp':>8}  {'corr eta':>9}  "
           f"{'RMSE u':>11}  {'corr u':>8}  {'wet':>5}")
    print("\n" + hdr)
    print("-" * len(hdr))
    for t_snap in snap_times:
        idx = np.argmin(np.abs(t_vals - t_snap))
        m   = wet[idx]
        if m.sum() < 3:
            continue
        re_, ce_ = _stats(ETA[idx], ETA_E[idx], m)
        ru_, cu_ = _stats(U[idx],   U_E[idx],   m)
        row = (f"{t_vals[idx]:>8.2f}  {re_:>11.4e}  {re_/A_REF:>8.4f}  {ce_:>9.4f}  "
               f"{ru_:>11.4e}  {cu_:>8.4f}  {m.sum():>5}")
        print(row); rows.append(row)

    # ---- global metrics ------------------------------------------------
    rmse_eta, corr_eta = _stats(ETA, ETA_E, wet)
    rmse_u,   corr_u   = _stats(U,   U_E,   wet)
    print(f"\nGlobal (wet points only):")
    print(f"  eta : RMSE = {rmse_eta:.4e} m   ({100*rmse_eta/A_REF:.2f}% of amplitude)"
          f"   Pearson r = {corr_eta:.4f}")
    print(f"  u   : RMSE = {rmse_u:.4e} m/s                       "
          f"   Pearson r = {corr_u:.4f}")

    # ---- maximum run-up -------------------------------------------------
    # Exact: sweep lambda over one period at sigma = 0 (the shoreline).
    lam_s = np.linspace(0.0, 2*np.pi/CG_W, 2001)
    _, eta_hat_s, *_ = _cg_state(np.zeros_like(lam_s), lam_s, CG_A)
    R_exact = float(np.max(CG_S * CG_L0 * eta_hat_s))

    # PINN: highest wet point over the whole run, converted to elevation.
    H_FIELD = bottom_depth(x_vals)[None, :] + ETA
    wet_p   = H_FIELD > 1e-3
    R_pinn  = float(CG_S * (x_vals[np.any(wet_p, axis=0)].max() - CG_XS)) \
              if wet_p.any() else np.nan

    print(f"\nMaximum run-up (vertical, above still water):")
    print(f"  exact = {R_exact:.4f} m")
    print(f"  PINN  = {R_pinn:.4f} m   "
          f"(relative error {100*abs(R_pinn-R_exact)/R_exact:.1f}%)")
    print(f"  model gap floor: CG is non-dispersive, kd={2*np.pi/cfg['wave_period']/np.sqrt(G*H0)*H0:.3f} "
          f"→ ~{(2*np.pi/cfg['wave_period']/np.sqrt(G*H0)*H0)**2/3*100:.2f}% expected")

    # ---- error field ----------------------------------------------------
    ERR = np.where(wet, np.abs(ETA - ETA_E), np.nan)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    im = axes[0].pcolormesh(x_vals, t_vals, ERR, cmap="magma", shading="auto")
    plt.colorbar(im, ax=axes[0], label=r"$|\eta_{\rm PINN}-\eta_{\rm exact}|$ [m]")
    axes[0].axvline(CG_XS, color="cyan", ls=":", lw=1)
    axes[0].set(title="Absolute error in η (white = dry)", xlabel="x [m]", ylabel="t [s]")

    with np.errstate(invalid="ignore"):
        err_t = np.nanmean(ERR, axis=1)
    axes[1].plot(t_vals, err_t / A_REF, color="crimson")
    axes[1].set(title="Mean |error| over x, normalised by amplitude",
                xlabel="t [s]", ylabel="mean |Δη| / a")
    axes[1].grid(True, alpha=0.3)
    plt.suptitle("PINN vs exact Carrier–Greenspan", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "error_vs_exact.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ---- save -----------------------------------------------------------
    with open(os.path.join(SAVE_DIR, "metrics.txt"), "w", encoding="utf-8") as f:
        f.write(f"Case : {cfg['label']}\n")
        f.write(f"Ref  : {cfg['ref']}\n")
        f.write(f"Reference: EXACT Carrier-Greenspan (hodograph), a={A_REF} m, "
                f"T={cfg['wave_period']} s, slope={CG_S:.4f}\n\n")
        f.write(hdr + "\n" + "\n".join(rows) + "\n\n")
        f.write(f"Global eta: RMSE={rmse_eta:.6e} ({100*rmse_eta/A_REF:.3f}% of a) "
                f"r={corr_eta:.6f}\n")
        f.write(f"Global u  : RMSE={rmse_u:.6e} r={corr_u:.6f}\n")
        f.write(f"Runup exact={R_exact:.6f} m  PINN={R_pinn:.6f} m\n")
    print(f"\nMetrics saved → {os.path.join(SAVE_DIR, 'metrics.txt')}")

else:
    print(f"No exact solution available for case '{ACTIVE_CASE}'.")
    print("Only Carrier-Greenspan has a closed-form reference in this notebook.")
    print("The Synolakis and Beji-Battjes cases need digitised laboratory data")
    print("to be validated quantitatively; the snapshots above are visual only.")

## 18. Save Weights

In [ ]:
wpath = os.path.join(SAVE_DIR, "pinn_weights.weights.h5")
model.save_weights(wpath)
print(f"Weights saved → {wpath}")
print(f"All outputs in: {SAVE_DIR}/")
print("Done ✓")